# 23 - Typed Tool Contracts, Action Safety & Audit Trails
**Stack:** Pydantic v2 · Python 3.12 · LangGraph  
**Agent OS pillar:** Typed tools and action contracts  

The central challenge of agentic AI in production: LLMs are non-deterministic.
Tools that write to systems of record must be deterministic and safe regardless
of what the LLM decides.

This notebook builds the **typed contract layer** between non-deterministic LLM
reasoning and deterministic system writes:
- **Typed tool schemas** - Pydantic v2 models as the single source of truth
- **Precondition validators** - check business invariants before side effects
- **Scoped permission model** - agents get least-privilege access by role
- **Idempotency enforcement** - every write is safe to retry
- **Audit log** - immutable record of every tool call
- **Rollback stubs** - compensating transactions for reversible actions

| Section | Topic |
|---|---|
| 1 | Why typed contracts matter |
| 2 | Pydantic v2 tool schemas |
| 3 | Precondition validators |
| 4 | Scoped permission model |
| 5 | Idempotency enforcement |
| 6 | Immutable audit log |
| 7 | Rollback / compensating transactions |
| 8 | Full demo: all layers together |


In [ ]:
from __future__ import annotations
import hashlib, json, time, uuid
from datetime import datetime
from enum import Enum
from typing import Any, Callable, Optional
from pydantic import BaseModel, Field, field_validator, model_validator
from functools import wraps

print('Imports OK')


## 1. Why Typed Contracts Matter

Without contracts, a tool call is just a dict being passed around:
```python
# Dangerous: no validation, no audit, no rollback
def dispatch_technician(data: dict):
    db.execute(f"UPDATE jobs SET tech_id = {data['tech_id']}")
```

With typed contracts:
```python
@governed_tool(permission=Permission.DISPATCH_WRITE, reversible=True)
def dispatch_technician(request: DispatchRequest) -> DispatchResult:
    ...
```

The contract layer is what separates a demo agent from a production agent.
Every field is validated, every call is audited, every write is idempotent.

In [ ]:
class JobStatus(str, Enum):
    SCHEDULED   = 'scheduled'
    IN_PROGRESS = 'in_progress'
    COMPLETED   = 'completed'
    CANCELLED   = 'cancelled'

class Permission(str, Enum):
    JOBS_READ      = 'jobs:read'
    JOBS_WRITE     = 'jobs:write'
    DISPATCH_READ  = 'dispatch:read'
    DISPATCH_WRITE = 'dispatch:write'
    INVOICE_READ   = 'invoice:read'
    INVOICE_WRITE  = 'invoice:write'
    CUSTOMER_READ  = 'customer:read'

class AgentRole(str, Enum):
    DISPATCHER  = 'dispatcher'
    CSR         = 'csr'
    FIELD_AGENT = 'field_agent'
    READ_ONLY   = 'read_only'

ROLE_PERMISSIONS: dict[AgentRole, set[Permission]] = {
    AgentRole.DISPATCHER: {
        Permission.JOBS_READ, Permission.JOBS_WRITE,
        Permission.DISPATCH_READ, Permission.DISPATCH_WRITE,
        Permission.CUSTOMER_READ,
    },
    AgentRole.CSR: {
        Permission.JOBS_READ, Permission.CUSTOMER_READ,
        Permission.INVOICE_READ,
    },
    AgentRole.FIELD_AGENT: {
        Permission.JOBS_READ, Permission.JOBS_WRITE,
        Permission.INVOICE_READ,
    },
    AgentRole.READ_ONLY: {
        Permission.JOBS_READ, Permission.CUSTOMER_READ,
        Permission.INVOICE_READ, Permission.DISPATCH_READ,
    },
}

print('Permission model defined.')
print(f'Dispatcher permissions: {sorted(p.value for p in ROLE_PERMISSIONS[AgentRole.DISPATCHER])}')
print(f'CSR permissions:        {sorted(p.value for p in ROLE_PERMISSIONS[AgentRole.CSR])}')


## 2. Pydantic v2 Tool Schemas

Every tool gets a frozen **request model** and a **result model**.
These are the typed contracts. The LLM must produce valid request payloads;
the tool must return valid result payloads. `frozen=True` means no mutation after creation.

In [ ]:
class DispatchRequest(BaseModel):
    model_config = {'frozen': True}

    idempotency_key: str = Field(
        description='Unique key for this dispatch. Same key = safe to retry.',
        min_length=8, max_length=128,
    )
    job_id: str          = Field(pattern=r'^JOB-\d+$')
    technician_id: str   = Field(pattern=r'^TECH-\d+$')
    tenant_id: str
    notes: Optional[str] = Field(default=None, max_length=500)

    @model_validator(mode='after')
    def idempotency_key_scoped_to_tenant(self):
        if not self.idempotency_key.startswith(self.tenant_id):
            raise ValueError(
                f'Idempotency key must be prefixed with tenant_id to prevent '
                f'cross-tenant key collisions. Got: {self.idempotency_key[:20]}'
            )
        return self

class DispatchResult(BaseModel):
    model_config = {'frozen': True}

    success: bool
    idempotency_key: str
    job_id: str
    technician_id: str
    was_duplicate: bool      = False
    dispatch_id: Optional[str] = None
    error: Optional[str]     = None
    processed_at: float      = Field(default_factory=time.time)

# Test schema validation
try:
    req = DispatchRequest(
        idempotency_key='TENANT_A:d1',
        job_id='JOB-001', technician_id='TECH-042', tenant_id='TENANT_A',
    )
    print('Valid DispatchRequest:', req.model_dump())
except Exception as e:
    print(f'ERROR: {e}')

try:
    bad = DispatchRequest(
        idempotency_key='bad-key',
        job_id='JOB-001', technician_id='TECH-042', tenant_id='TENANT_A',
    )
except Exception as e:
    print(f'Correctly rejected bad key: {type(e).__name__}')


## 3. Precondition Validators

Before executing a write, check business invariants. These prevent an agent from doing something technically valid but operationally wrong (e.g., dispatching an unavailable technician, or accessing another tenant's job).

In [ ]:
class PreconditionError(Exception):
    def __init__(self, condition: str, message: str):
        self.condition = condition
        self.message = message
        super().__init__(f'[{condition}] {message}')

class Precondition:
    def __init__(self, name: str, check_fn: Callable, message_fn: Callable):
        self.name = name
        self.check_fn = check_fn
        self.message_fn = message_fn
    def validate(self, *args, **kwargs):
        if not self.check_fn(*args, **kwargs):
            raise PreconditionError(self.name, self.message_fn(*args, **kwargs))

# Simulated state store
JOBS: dict[str, dict] = {
    'JOB-001': {'status': 'scheduled',   'tenant_id': 'TENANT_A', 'tech_id': None},
    'JOB-002': {'status': 'in_progress', 'tenant_id': 'TENANT_A', 'tech_id': 'TECH-010'},
    'JOB-003': {'status': 'completed',   'tenant_id': 'TENANT_A', 'tech_id': 'TECH-005'},
    'JOB-099': {'status': 'scheduled',   'tenant_id': 'TENANT_B', 'tech_id': None},
}
TECHS: dict[str, dict] = {
    'TECH-042': {'available': True,  'tenant_id': 'TENANT_A'},
    'TECH-010': {'available': False, 'tenant_id': 'TENANT_A'},
    'TECH-005': {'available': True,  'tenant_id': 'TENANT_A'},
}

DISPATCH_PRECONDITIONS = [
    Precondition('job_exists',
        lambda r: r.job_id in JOBS,
        lambda r: f'Job {r.job_id} does not exist'),
    Precondition('job_tenant_scoped',
        lambda r: JOBS.get(r.job_id, {}).get('tenant_id') == r.tenant_id,
        lambda r: f'Job {r.job_id} belongs to a different tenant'),
    Precondition('job_is_schedulable',
        lambda r: JOBS.get(r.job_id, {}).get('status') == 'scheduled',
        lambda r: f'Job {r.job_id} status is {JOBS.get(r.job_id, {}).get("status")} -- cannot dispatch'),
    Precondition('tech_exists',
        lambda r: r.technician_id in TECHS,
        lambda r: f'Technician {r.technician_id} does not exist'),
    Precondition('tech_available',
        lambda r: TECHS.get(r.technician_id, {}).get('available', False),
        lambda r: f'Technician {r.technician_id} is not available'),
    Precondition('tech_tenant_scoped',
        lambda r: TECHS.get(r.technician_id, {}).get('tenant_id') == r.tenant_id,
        lambda r: f'Technician {r.technician_id} belongs to a different tenant'),
]

def run_preconditions(preconditions, request):
    for pre in preconditions:
        pre.validate(request)

test_cases = [
    ('Valid dispatch',          DispatchRequest(idempotency_key='TENANT_A:t1', job_id='JOB-001', technician_id='TECH-042', tenant_id='TENANT_A')),
    ('Tech unavailable',        DispatchRequest(idempotency_key='TENANT_A:t2', job_id='JOB-001', technician_id='TECH-010', tenant_id='TENANT_A')),
    ('Job already in progress', DispatchRequest(idempotency_key='TENANT_A:t3', job_id='JOB-002', technician_id='TECH-042', tenant_id='TENANT_A')),
    ('Cross-tenant job access', DispatchRequest(idempotency_key='TENANT_A:t4', job_id='JOB-099', technician_id='TECH-042', tenant_id='TENANT_A')),
]

for label, req in test_cases:
    try:
        run_preconditions(DISPATCH_PRECONDITIONS, req)
        print(f'  PASS  {label}')
    except PreconditionError as e:
        print(f'  BLOCK {label}: {e}')


## 4. Immutable Audit Log

Every tool call -- success or failure -- is appended to an immutable log. This is the evidence trail for compliance, debugging, and tenant billing.

In [ ]:
class AuditEntry(BaseModel):
    model_config = {'frozen': True}
    entry_id: str        = Field(default_factory=lambda: str(uuid.uuid4()))
    ts: float            = Field(default_factory=time.time)
    tenant_id: str
    agent_role: str
    tool_name: str
    idempotency_key: str
    request_hash: str
    outcome: str
    error_detail: Optional[str] = None
    duration_ms: float   = 0.0
    was_duplicate: bool  = False

_AUDIT_LOG: list[AuditEntry] = []

def audit(entry: AuditEntry):
    _AUDIT_LOG.append(entry)

def get_audit_log(tenant_id=None):
    if tenant_id:
        return [e for e in _AUDIT_LOG if e.tenant_id == tenant_id]
    return list(_AUDIT_LOG)

print('Audit log initialized')


## 5. @governed_tool Decorator

Wires everything together: permission check -> precondition check -> idempotency check -> execute -> audit log.

In [ ]:
_IDEMPOTENCY_STORE: dict[str, Any] = {}

def governed_tool(permission: Permission, preconditions=None, reversible=False):
    preconditions = preconditions or []
    def decorator(fn):
        @wraps(fn)
        def wrapper(request: BaseModel, *, agent_role: AgentRole, tenant_id: str):
            t0 = time.time()
            ikey = getattr(request, 'idempotency_key', str(uuid.uuid4()))
            req_hash = hashlib.sha256(request.model_dump_json().encode()).hexdigest()

            # 1. Permission
            if permission not in ROLE_PERMISSIONS.get(agent_role, set()):
                audit(AuditEntry(tenant_id=tenant_id, agent_role=agent_role.value,
                    tool_name=fn.__name__, idempotency_key=ikey, request_hash=req_hash,
                    outcome='permission_denied',
                    error_detail=f'Role {agent_role.value} lacks {permission.value}'))
                raise PermissionError(f'Role {agent_role.value!r} lacks permission {permission.value!r}')

            # 2. Idempotency
            scoped = f'{tenant_id}:{ikey}'
            if scoped in _IDEMPOTENCY_STORE:
                audit(AuditEntry(tenant_id=tenant_id, agent_role=agent_role.value,
                    tool_name=fn.__name__, idempotency_key=ikey, request_hash=req_hash,
                    outcome='success', duration_ms=(time.time()-t0)*1000, was_duplicate=True))
                print(f'  [idempotency] returning cached result for {ikey}')
                return _IDEMPOTENCY_STORE[scoped]

            # 3. Preconditions
            try:
                run_preconditions(preconditions, request)
            except PreconditionError as e:
                audit(AuditEntry(tenant_id=tenant_id, agent_role=agent_role.value,
                    tool_name=fn.__name__, idempotency_key=ikey, request_hash=req_hash,
                    outcome='precondition_failed', error_detail=str(e)))
                raise

            # 4. Execute
            try:
                result = fn(request)
                _IDEMPOTENCY_STORE[scoped] = result
                audit(AuditEntry(tenant_id=tenant_id, agent_role=agent_role.value,
                    tool_name=fn.__name__, idempotency_key=ikey, request_hash=req_hash,
                    outcome='success', duration_ms=(time.time()-t0)*1000))
                return result
            except Exception as e:
                audit(AuditEntry(tenant_id=tenant_id, agent_role=agent_role.value,
                    tool_name=fn.__name__, idempotency_key=ikey, request_hash=req_hash,
                    outcome='error', error_detail=str(e)))
                raise
        wrapper.permission = permission
        wrapper.reversible = reversible
        return wrapper
    return decorator

print('@governed_tool decorator defined')


In [ ]:
@governed_tool(permission=Permission.DISPATCH_WRITE,
               preconditions=DISPATCH_PRECONDITIONS, reversible=True)
def dispatch_technician(request: DispatchRequest) -> DispatchResult:
    dispatch_id = f'DISP-{uuid.uuid4().hex[:8].upper()}'
    JOBS[request.job_id]['tech_id'] = request.technician_id
    JOBS[request.job_id]['status']  = 'in_progress'
    TECHS[request.technician_id]['available'] = False
    print(f'  [dispatch] {request.technician_id} -> {request.job_id} ({dispatch_id})')
    return DispatchResult(success=True, idempotency_key=request.idempotency_key,
        job_id=request.job_id, technician_id=request.technician_id, dispatch_id=dispatch_id)

print('=== Scenario 1: Valid dispatch ===')
r1 = dispatch_technician(
    DispatchRequest(idempotency_key='TENANT_A:d1', job_id='JOB-001', technician_id='TECH-042', tenant_id='TENANT_A'),
    agent_role=AgentRole.DISPATCHER, tenant_id='TENANT_A')
print(f'Result: success={r1.success}, dispatch_id={r1.dispatch_id}')

print('\n=== Scenario 2: Retry same key (idempotent) ===')
r1b = dispatch_technician(
    DispatchRequest(idempotency_key='TENANT_A:d1', job_id='JOB-001', technician_id='TECH-042', tenant_id='TENANT_A'),
    agent_role=AgentRole.DISPATCHER, tenant_id='TENANT_A')
print(f'Result: success={r1b.success} (cached)')

print('\n=== Scenario 3: CSR tries to dispatch (permission denied) ===')
try:
    dispatch_technician(
        DispatchRequest(idempotency_key='TENANT_A:d2', job_id='JOB-001', technician_id='TECH-042', tenant_id='TENANT_A'),
        agent_role=AgentRole.CSR, tenant_id='TENANT_A')
except PermissionError as e:
    print(f'Blocked: {e}')

print('\n=== Scenario 4: Cross-tenant access ===')
try:
    dispatch_technician(
        DispatchRequest(idempotency_key='TENANT_A:d3', job_id='JOB-099', technician_id='TECH-042', tenant_id='TENANT_A'),
        agent_role=AgentRole.DISPATCHER, tenant_id='TENANT_A')
except PreconditionError as e:
    print(f'Blocked: {e}')


In [ ]:
print('=== Audit Log (TENANT_A) ===\n')
for e in get_audit_log('TENANT_A'):
    ts = datetime.fromtimestamp(e.ts).strftime('%H:%M:%S')
    dup = ' [DUP]' if e.was_duplicate else ''
    print(f'  {ts} | {e.outcome:20s} | {e.tool_name:25s} | role={e.agent_role}{dup}')
    if e.error_detail:
        print(f'           >> {e.error_detail}')


## 6. Rollback / Compensating Transactions

For reversible actions, define a compensating transaction that undoes the side effect. This is the foundation of saga-pattern workflows.

In [ ]:
_ROLLBACK_REGISTRY: dict[str, Callable] = {}

def register_rollback(tool_name: str, fn: Callable):
    _ROLLBACK_REGISTRY[tool_name] = fn

def rollback(tool_name: str, **kwargs):
    fn = _ROLLBACK_REGISTRY.get(tool_name)
    if not fn:
        raise ValueError(f'No rollback for {tool_name}')
    return fn(**kwargs)

def _rollback_dispatch(job_id: str):
    job = JOBS.get(job_id)
    if job and job.get('status') == 'in_progress':
        tech = job['tech_id']
        job['status'] = 'scheduled'
        job['tech_id'] = None
        if tech and tech in TECHS:
            TECHS[tech]['available'] = True
        print(f'  [rollback] Undid dispatch on {job_id}, freed {tech}')
        return True
    return False

register_rollback('dispatch_technician', _rollback_dispatch)

print('Before:', {k: v for k, v in JOBS['JOB-001'].items()})
rollback('dispatch_technician', job_id='JOB-001')
print('After: ', {k: v for k, v in JOBS['JOB-001'].items()})


In [ ]:
print('=' * 60)
print('Notebook 23: Typed Tool Contracts, Action Safety & Audit Trails')
print('=' * 60)
print('''
Demonstrated:
  [x] Pydantic v2 frozen models as typed tool contracts
  [x] Cross-tenant write prevention via idempotency key prefix
  [x] Named precondition validators (status, availability, tenant scope)
  [x] Scoped permission model (role -> permission set)
  [x] Idempotency store -- safe retries, no double-writes
  [x] Immutable audit log -- every call recorded
  [x] Rollback / compensating transactions

Agent OS JD mapping:
  Typed tools and action contracts -> DispatchRequest/Result
  Precondition checks              -> DISPATCH_PRECONDITIONS
  Scoped permissions               -> ROLE_PERMISSIONS + @governed_tool
  Idempotency                      -> _IDEMPOTENCY_STORE
  Audit trails                     -> AuditEntry + _AUDIT_LOG
  Rollback                         -> register_rollback/_rollback_dispatch
''')
